# Tic-Tac-Toe với Alpha-Beta Pruning & Heuristic AI

**Đề tài 6** — mở rộng đề 5 (Minimax) bằng Alpha-Beta pruning.

Notebook này gói toàn bộ project để dễ thuyết trình:

- 3 thuật toán: **Minimax**, **Alpha-Beta**, **Alpha-Beta + Heuristic ordering**
- So sánh số node duyệt, thời gian, tỉ lệ thắng
- **Tương tác người vs AI** ngay trong notebook (widget)
- Vẽ **cây tìm kiếm 10–30 nodes** đúng yêu cầu đề bài
- Benchmark vòng tròn 3 AI

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import ipywidgets as widgets

# Tìm project root (chứa thư mục src/)
def find_root():
    cwd = os.getcwd()
    for _ in range(5):
        if os.path.isdir(os.path.join(cwd, 'src')):
            return cwd
        cwd = os.path.dirname(cwd)
    return os.getcwd()

ROOT = find_root()
sys.path.insert(0, ROOT)
print(f"Project root: {ROOT}")

from src import board as bd
from src import heuristics as heu
from src.algorithms import minimax, alpha_beta, heuristic_ai
from src.metrics import play_game, make_heuristic_ai, all_ai_callables

plt.rcParams['figure.dpi'] = 100


: 

## 1. Đặc tả bài toán (PEAS)

| Thành phần | Mô tả |
|---|---|
| **Performance** | Thắng (+1), Hòa (0), Thua (−1); phụ: số node duyệt, thời gian |
| **Environment** | Bàn cờ Tic-Tac-Toe 3×3 — quan sát đầy đủ, xác định, tĩnh, rời rạc, đa tác nhân cạnh tranh |
| **Actuators** | Đặt X hoặc O vào ô trống (r, c) |
| **Sensors** | Quan sát toàn bộ 9 ô của bàn cờ |

**Định nghĩa bài toán tìm kiếm:**

- **State**: ma trận 3×3, ô = 0 (trống) / 1 (X) / 2 (O)
- **Initial state**: bàn cờ trống, X đi trước
- **Actions**: tập ô trống (giảm dần)
- **Transition**: `RESULT(s, a)` = bàn cờ mới với ô `a` điền quân của player
- **Goal test**: 3 quân cùng hàng/cột/chéo, hoặc bàn cờ đầy
- **Path cost**: không cần (zero-sum game)

In [ ]:
def render_board_html(state, last_move=None):
    sym = {1: ('X', '#1f77b4'), 2: ('O', '#d62728'), 0: ('·', '#999')}
    cells_html = []
    for r in range(3):
        for c in range(3):
            v = int(state[r, c])
            ch, col = sym[v]
            bg = '#fffbe6' if last_move == (r, c) else 'white'
            cells_html.append(
                f'<td style="width:60px;height:60px;text-align:center;'
                f'font-size:32px;font-weight:bold;color:{col};'
                f'border:2px solid #444;background:{bg}">{ch}</td>'
            )
    rows = ''.join(
        f'<tr>{"".join(cells_html[r*3:(r+1)*3])}</tr>' for r in range(3)
    )
    return f'<table style="border-collapse:collapse;margin:8px 0">{rows}</table>'


def ai_factory(name, heuristic_name='h2_threat'):
    if name == 'minimax':
        return minimax.choose_move
    if name == 'alpha_beta':
        return alpha_beta.choose_move
    if name == 'heuristic_ai':
        return make_heuristic_ai(heuristic_name)
    raise ValueError(name)


def current_player(state):
    n_x = int(np.sum(state == bd.PLAYER_X))
    n_o = int(np.sum(state == bd.PLAYER_O))
    return bd.PLAYER_X if n_x <= n_o else bd.PLAYER_O


## 2. Demo tương tác — Người vs AI

Chọn AI cho mỗi phe, chọn heuristic, click ô trống để đi. AI tự đi nước kế tiếp và in ra số node + thời gian.

In [ ]:
GAME = {'state': None, 'log': [], 'last': None}

ai_x_dd = widgets.Dropdown(
    options=['alpha_beta', 'minimax', 'heuristic_ai'],
    value='alpha_beta', description='AI for X:',
)
ai_o_dd = widgets.Dropdown(
    options=['alpha_beta', 'minimax', 'heuristic_ai'],
    value='heuristic_ai', description='AI for O:',
)
h_dd = widgets.Dropdown(
    options=['h1_position', 'h2_threat'],
    value='h2_threat', description='Heuristic:',
)
mode_dd = widgets.Dropdown(
    options=['Human (X) vs AI (O)', 'AI (X) vs Human (O)', 'AI vs AI'],
    value='Human (X) vs AI (O)', description='Mode:',
)
reset_btn = widgets.Button(description='🔄 Reset', button_style='warning')
ai_step_btn = widgets.Button(description='▶ AI step', button_style='info')

status_html = widgets.HTML(value='<b>Status:</b> Press Reset')
board_html = widgets.HTML(value=render_board_html(bd.empty_board()))
log_html = widgets.HTML(value='<b>Move log:</b><br>')

cell_btns = []
for r in range(3):
    row = []
    for c in range(3):
        b = widgets.Button(
            description=' ', layout=widgets.Layout(width='64px', height='64px'),
            button_style='',
        )
        b.on_click(lambda _, r=r, c=c: on_cell(r, c))
        row.append(b)
    cell_btns.append(row)
cell_grid = widgets.VBox([widgets.HBox(row) for row in cell_btns])


def refresh_board():
    """Cập nhật hiển thị 9 nút + status."""
    st = GAME['state']
    for r in range(3):
        for c in range(3):
            v = int(st[r, c])
            if v == 0:
                cell_btns[r][c].description = ' '
                cell_btns[r][c].disabled = False
                cell_btns[r][c].style = widgets.ButtonStyle()
            else:
                cell_btns[r][c].description = bd.SYMBOLS[v]
                cell_btns[r][c].disabled = True
                color = '#1f77b4' if v == 1 else '#d62728'
                cell_btns[r][c].style = widgets.ButtonStyle(
                    font_weight='bold', button_color='white', text_color=color,
                )
            if GAME['last'] == (r, c):
                cell_btns[r][c].style = widgets.ButtonStyle(
                    font_weight='bold', button_color='#fffbe6',
                )
    log_html.value = '<b>Move log:</b><br>' + '<br>'.join(GAME['log'])


def get_human():
    m = mode_dd.value
    if m == 'AI vs AI':
        return None
    if m.startswith('Human (X)'):
        return bd.PLAYER_X
    return bd.PLAYER_O


def ai_turn():
    st = GAME['state']
    if bd.winner(st) != 0 or not np.any(st == bd.EMPTY):
        return False
    cur = current_player(st)
    name = ai_x_dd.value if cur == bd.PLAYER_X else ai_o_dd.value
    ai = ai_factory(name, h_dd.value)
    metrics = {'nodes_visited': 0}
    t0 = time.perf_counter()
    move = ai(st, cur, metrics)
    dt = (time.perf_counter() - t0) * 1000
    GAME['state'] = bd.apply_move(st, move, cur)
    GAME['last'] = move
    GAME['log'].append(
        f'  {bd.SYMBOLS[cur]} → ({move[0]+1},{move[1]+1})  '
        f'[{name}]  nodes={metrics["nodes_visited"]:,}  time={dt:.2f} ms'
    )
    return True


def end_check():
    w = bd.winner(GAME['state'])
    if w != 0:
        status_html.value = (
            f'<b>Status:</b> <span style="color:green;font-size:18px">'
            f'{bd.SYMBOLS[w]} wins!</span>'
        )
        return True
    if not np.any(GAME['state'] == bd.EMPTY):
        status_html.value = '<b>Status:</b> <span style="color:gray">Draw.</span>'
        return True
    return False


def on_cell(r, c):
    if get_human() is None:
        return  # AI vs AI
    st = GAME['state']
    if st[r, c] != bd.EMPTY:
        return
    human = get_human()
    if current_player(st) != human:
        return
    st = bd.apply_move(st, (r, c), human)
    GAME['state'] = st
    GAME['last'] = (r, c)
    GAME['log'].append(f'  {bd.SYMBOLS[human]} → ({r+1},{c+1})  [Human]')
    refresh_board()
    if end_check():
        return
    # AI đi tiếp
    if ai_turn():
        refresh_board()
        if end_check():
            return
    cp = current_player(GAME['state'])
    if get_human() == cp:
        status_html.value = f'<b>Status:</b> Your turn ({bd.SYMBOLS[cp]})'
    else:
        status_html.value = f'<b>Status:</b> AI ({bd.SYMBOLS[cp]}) thinking…'


def on_reset(_):
    GAME['state'] = bd.empty_board()
    GAME['log'] = []
    GAME['last'] = None
    refresh_board()
    if get_human() is None:
        status_html.value = '<b>Status:</b> AI vs AI — bấm ▶ AI step'
    else:
        if current_player(GAME['state']) == get_human():
            status_html.value = f'<b>Status:</b> Your turn ({bd.SYMBOLS[get_human()]})'
        else:
            status_html.value = f'<b>Status:</b> AI starts ({bd.SYMBOLS[bd.PLAYER_X]})'
            if ai_turn():
                refresh_board()
                if not end_check():
                    status_html.value = f'<b>Status:</b> Your turn'


def on_ai_step(_):
    if ai_turn():
        refresh_board()
        end_check()


reset_btn.on_click(on_reset)
ai_step_btn.on_click(on_ai_step)
GAME['state'] = bd.empty_board()
refresh_board()
status_html.value = '<b>Status:</b> Press 🔄 Reset'

display(widgets.VBox([
    widgets.HBox([ai_x_dd, ai_o_dd]),
    widgets.HBox([h_dd, mode_dd]),
    widgets.HBox([reset_btn, ai_step_btn]),
    cell_grid,
    status_html,
    log_html,
]))


## 3. Hàm heuristic

Heuristic **chỉ dùng để sắp xếp thứ tự nước đi** (move ordering) — không cắt tỉa kết quả. Vì vậy chúng không cần thoả `admissible/consistent` như trong A*.

**h1 — Position score** (theo AsadJatoi 2023):
Tổng trọng số vị trí. Center = 3, góc = 2, cạnh = 1.

**h2 — Threat / line-formation score** (theo Don Yitong Ma 2023):
Quét 8 đường (3 hàng + 3 cột + 2 chéo), chấm điểm theo số quân thân / địch.

In [ ]:
state = bd.empty_board()
state[1, 1] = bd.PLAYER_X
state[0, 0] = bd.PLAYER_X
state[2, 2] = bd.PLAYER_O
print(render_board_html(state, last_move=(2, 2)))
print(f"h1 (X) = {heu.h1_position_score(state, bd.PLAYER_X)}")
print(f"h1 (O) = {heu.h1_position_score(state, bd.PLAYER_O)}")
print(f"h2 (X) = {heu.h2_threat_score(state, bd.PLAYER_X)}")
print(f"h2 (O) = {heu.h2_threat_score(state, bd.PLAYER_O)}")


## 4. Demo 1 ván AI vs AI

Chạy 1 ván Alpha-Beta vs Heuristic (cả 2 đều optimal → thường hòa) và in từng nước đi.

In [ ]:
def replay_game(ai_x_name, ai_o_name, heuristic='h2_threat', max_moves=20):
    state = bd.empty_board()
    ais = {1: ai_factory(ai_x_name, heuristic),
           2: ai_factory(ai_o_name, heuristic)}
    cur = bd.PLAYER_X
    print(f"{ai_x_name} (X) vs {ai_o_name} (O)\n")
    print(render_board_html(state))
    for step in range(max_moves):
        if bd.winner(state) != 0 or not np.any(state == bd.EMPTY):
            break
        m = {'nodes_visited': 0}
        t0 = time.perf_counter()
        move = ais[cur](state, cur, m)
        dt = (time.perf_counter() - t0) * 1000
        state = bd.apply_move(state, move, cur)
        print(
            f"  Move {step+1}: {bd.SYMBOLS[cur]} → ({move[0]+1},{move[1]+1})  "
            f"nodes={m['nodes_visited']:,}  time={dt:.1f} ms"
        )
        print(render_board_html(state, last_move=move))
        cur = bd.other(cur)
    w = bd.winner(state)
    if w == 0:
        print('Draw.')
    else:
        print(f'{bd.SYMBOLS[w]} wins!')


replay_game('alpha_beta', 'heuristic_ai', 'h2_threat')


## 5. Cây tìm kiếm (10–30 nodes)

Đề bài yêu cầu cây tìm kiếm **10–30 nodes** trong phần 4. Hình 586 nodes từ full depth=3 vượt giới hạn, nên ta vẽ cây con: X đi 3 nước đầu, mỗi nhánh O đi 8 nước, 1 nhánh mở rộng thêm 1 lớp. **Tổng 25 nodes.**

In [ ]:
# Vẽ cây nhỏ (10-30 nodes) — depth 0..3, 25 nodes
class TNode:
    __slots__ = ('state', 'player', 'score', 'move', 'children')
    def __init__(self, state, player, score=0, move=None):
        self.state = state
        self.player = player
        self.score = score
        self.move = move
        self.children = []

root = TNode(bd.empty_board(), bd.PLAYER_X, score=0, move=None)
x_moves = [(0, 0), (1, 1), (0, 2)]
opp = bd.other(bd.PLAYER_X)
x_children = []
for m1 in x_moves:
    s1 = bd.apply_move(root.state, m1, bd.PLAYER_X)
    n1 = TNode(s1, bd.PLAYER_X, score=0, move=m1)
    root.children.append(n1)
    x_children.append(n1)
for c1 in x_children[:2]:  # 2 nhánh X đầu có O
    for m2 in bd.legal_moves(c1.state):
        s2 = bd.apply_move(c1.state, m2, opp)
        n2 = TNode(s2, opp, score=0, move=m2)
        c1.children.append(n2)
        # Mở rộng thêm 1 nước ở 5 nhánh O đầu
for c2 in root.children[0].children[:5]:
    for m3 in bd.legal_moves(c2.state)[:1]:
        s3 = bd.apply_move(c2.state, m3, bd.PLAYER_X)
        n3 = TNode(s3, bd.PLAYER_X, score=0, move=m3)
        c2.children.append(n3)
total = 1 + sum(len(c.children) for c in root.children) + \
        sum(len(c2.children) for c1 in root.children for c2 in c1.children)
print(f'Total nodes: {total}')

# Layout: tầng đầu dùng grid cho gọn, không cần layout thuật toán.
fig, ax = plt.subplots(figsize=(15, 8))
sym = {1: ('X', '#1a5276'), 2: ('O', '#922b21'), 0: ('·', '#bbb')}

def draw_node(ax, x, y, state, score, w=0.55, h=0.42, is_root=False):
    fc = '#f8f9fa' if is_root else 'white'
    rect = plt.Rectangle((x-w/2, y-h/2), w, h, facecolor=fc,
                         edgecolor='#444', linewidth=1.2, zorder=1)
    ax.add_patch(rect)
    # Vẽ X/O bằng text
    for r in range(3):
        for c in range(3):
            v = int(state[r, c])
            if v == 0: continue
            ch, col = sym[v]
            ax.text(x + (c-1)*0.10, y + (1-r)*0.10 + 0.07, ch,
                    ha='center', va='center', fontsize=12,
                    fontweight='bold', color=col, zorder=3)
    ax.text(x, y-0.28, f'[{score:+d}]', ha='center', va='center',
            fontsize=8, color='#555', zorder=3)

def draw_edge(ax, x1, y1, x2, y2, label):
    ax.annotate('', xy=(x2, y2+0.21), xytext=(x1, y1-0.21),
                arrowprops=dict(arrowstyle='->', color='#555', lw=0.8))
    ax.text((x1+x2)/2 + 0.03, (y1+y2)/2, label,
            fontsize=7, color='#555', zorder=2)

max_x = 16.0
depth_y = {0: 3, 1: 2, 2: 1, 3: 0}
node_pos = {}
# Root
draw_node(ax, max_x/2, depth_y[0], root.state, 0, is_root=True)
node_pos[id(root)] = (max_x/2, depth_y[0])
# Depth 1: 3 nhánh X
spacing1 = max_x / (len(root.children) + 1)
for i, c1 in enumerate(root.children):
    x = spacing1 * (i+1)
    y = depth_y[1]
    draw_node(ax, x, y, c1.state, c1.score)
    node_pos[id(c1)] = (x, y)
    draw_edge(ax, *node_pos[id(root)], x, y, f"({c1.move[0]+1},{c1.move[1]+1})")
# Depth 2: O ở 2 nhánh X đầu, mỗi nhánh 8 node
for c1 in root.children[:2]:
    px, py = node_pos[id(c1)]
    n = len(c1.children)
    spread = 1.6
    for i, c2 in enumerate(c1.children):
        x = px + (i - (n-1)/2) * (spread / max(n-1, 1))
        y = depth_y[2]
        draw_node(ax, x, y, c2.state, c2.score)
        node_pos[id(c2)] = (x, y)
        draw_edge(ax, px, py, x, y, f"({c2.move[0]+1},{c2.move[1]+1})")
# Depth 3: X mở rộng 5 nhánh O đầu của nhánh X[0]
for c2 in root.children[0].children[:5]:
    px, py = node_pos[id(c2)]
    for c3 in c2.children:
        x = px + 0.5
        y = depth_y[3]
        draw_node(ax, x, y, c3.state, c3.score)
        draw_edge(ax, px, py, x, y, f"({c3.move[0]+1},{c3.move[1]+1})")
ax.set_xlim(-0.5, max_x + 0.5)
ax.set_ylim(-0.6, 3.6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(f'Partial Minimax Search Tree — {total} nodes (depth 0-3)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Benchmark vòng tròn

Chạy 3 AI đấu vòng tròn (mỗi cặp N ván, đổi phe để cân bằng). Minimax chậm → giảm số ván cho cặp có Minimax.

In [ ]:
N_FULL = 6     # cho cặp có Minimax (chậm)
N_FAST = 20    # cho cặp chỉ Alpha-Beta / Heuristic

matchups = [
    ('minimax', 'alpha_beta', N_FULL),
    ('minimax', 'heuristic_ai', N_FULL),
    ('alpha_beta', 'heuristic_ai', N_FAST),
]

summary = []
for ax_name, ao_name, n in matchups:
    print(f'\n=== {ax_name} (X) vs {ao_name} (O) — {n} games ===')
    nodes_list = []
    time_list = []
    wins = {ax_name: 0, ao_name: 0, 'draw': 0}
    t0 = time.perf_counter()
    for i in range(n):
        # ván lẻ đổi phe
        if i % 2 == 1:
            x_name, o_name = ao_name, ax_name
        else:
            x_name, o_name = ax_name, ao_name
        ai_x = ai_factory(x_name, 'h2_threat')
        ai_o = ai_factory(o_name, 'h2_threat')
        r = play_game(ai_x, ai_o)
        nodes_list.append(r.total_nodes)
        time_list.append(r.total_time_ms)
        if r.winner == bd.PLAYER_X:
            wins[x_name] += 1
        elif r.winner == bd.PLAYER_O:
            wins[o_name] += 1
        else:
            wins['draw'] += 1
    dt = time.perf_counter() - t0
    avg_n = int(np.mean(nodes_list))
    avg_t = np.mean(time_list)
    print(f'  wins={wins}  avg_nodes={avg_n:,}  avg_time={avg_t:.1f} ms  wall={dt:.1f}s')
    summary.append({
        'matchup': f'{ax_name} vs {ao_name}',
        'n': n,
        'avg_nodes': avg_n,
        'avg_time_ms': round(avg_t, 2),
        'wins': wins,
    })
print('\n--- Summary ---')
for s in summary:
    print(f"  {s['matchup']:30s}  nodes={s['avg_nodes']:>9,}  time={s['avg_time_ms']:>8.1f} ms")


## 7. Biểu đồ so sánh

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
names = [s['matchup'] for s in summary]
nodes = [s['avg_nodes'] for s in summary]
times = [s['avg_time_ms'] for s in summary]
colors = ['#4c72b0', '#dd8452', '#55a868']
axes[0].bar(names, nodes, color=colors)
axes[0].set_ylabel('Avg nodes visited / game')
axes[0].set_title('Số node duyệt (ít = tốt)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, axis='y', alpha=0.3)
axes[1].bar(names, times, color=colors)
axes[1].set_ylabel('Avg time (ms) / game')
axes[1].set_title('Thời gian chạy (ms)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Kết luận

- **Alpha-Beta pruning** cắt giảm số nút từ O(b^d) xuống O(b^(d/2)) trong trường hợp tốt nhất
- **Move ordering theo heuristic** giảm thêm ~15–20% số nút
- Trên Tic-Tac-Toe 3×3, cả 3 AI đều optimal → 100% hòa khi AI-vs-AI
- Trên **Caro / Gomoku** (bàn cờ lớn hơn) sự khác biệt sẽ rõ rệt hơn nhiều

**Hướng phát triển:**

- Mở rộng sang Caro (NxN)
- Transposition table (zobrist hashing)
- Iterative deepening
